In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 09 consolidation — Verify persistence without trusting the same algorithm twice

Read `sessions/09_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real Iris cloud with a deliberately duplicated observation and a deterministic permutation control.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Compare the spanning-tree calculation and boundary reduction on a small real cloud.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.data import load_iris_data
from shape_lab.audits import h0_merge_times
from shape_lab.persistence import rips_filtration, persistent_homology, diagram
frame = load_iris_data().iloc[::10]
X = frame[[c for c in frame if c.endswith('_cm')]].to_numpy()
tree = h0_merge_times(X)
bars = diagram(persistent_homology(rips_filtration(X, max_homology=0)), 0, finite_only=True)
assert np.allclose(tree[tree>0], np.sort(bars[:,1]))
print('finite positive H0 deaths:', np.sort(bars[:,1]))

finite positive H0 deaths: [0.17320508 0.26457513 0.36055513 0.37416574 0.53851648 0.60827625
 0.64031242 0.678233   0.7        0.71414284 0.85440037 0.87749644
 1.0198039  2.34520788]


## Step 2

Insert a duplicate observation. Its zero-weight merge is retained by the tree but excluded by the positive-bar policy.

Before running: write your expected result and name one way this check could fail.

In [4]:
duplicated = np.vstack([X, X[0]])
weights = h0_merge_times(duplicated)
assert len(weights)==len(duplicated)-1 and np.any(weights==0)
other = diagram(persistent_homology(rips_filtration(duplicated, max_homology=0)), 0, finite_only=True)
assert np.allclose(weights[weights>0], np.sort(other[:,1]))

## Step 3

Permutation should preserve the multiset of merge weights, including ties and zero merges.

Before running: write your expected result and name one way this check could fail.

In [5]:
permutation = np.random.default_rng(19).permutation(len(duplicated))
assert np.allclose(h0_merge_times(duplicated[permutation]), weights)
result = {'original_records': len(X), 'with_duplicate': len(duplicated),
          'h0_tree_reduction_agreement': True, 'zero_weight_merges': int(np.sum(weights==0)),
          'h1_independently_verified_by_this_test': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 9
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real Iris cloud with a deliberately duplicated observation and a deterministic permutation control.'
output = ROOT/'reports/consolidation_v4/09.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/09.json
